# Phase 2 & 3: Deep Learning (Keras Tokenizer + Bidirectional LSTM)

This notebook covers the implementation of a Deep Learning architecture (LSTM RNN) for sentiment classification on the **BrandPulse AI** platform.

In [6]:
import os
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SpatialDropout1D, Bidirectional, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# add parent dir to path so we can import our preprocessing script
import sys
sys.path.append('..')
from preprocessing import clean_tweet

print("TensorFlow Version:", tf.__version__)
print("Imports completed successfully.")

ModuleNotFoundError: No module named 'tensorflow'

## 1. Data Loading and Cleaning

We reload the `Tweets.csv` dataset and clean it.

In [ ]:
data_path = '../data/Tweets.csv'
if not os.path.exists(data_path):
    raise FileNotFoundError("Please run notebook 01 first to download the dataset.")

df = pd.read_csv(data_path)
print("Cleaning tweets...")
df['clean_text'] = df['text'].apply(clean_tweet)
df = df[df['clean_text'].str.strip() != '']
print(f"Size after cleaning: {df.shape[0]} tweets.")

Cleaning tweets...


NameError: name 'clean_tweet' is not defined

## 2. Label Encoding & Tokenization

The string labels need to be converted to numeric categories for our network's Softmax output.

In [ ]:
# map labels
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label'] = df['airline_sentiment'].map(label_map)

# Tokenize text
max_features = 10000  # max unique words
tokenizer = Tokenizer(num_words=max_features, split=' ', oov_token='<OOV>')
tokenizer.fit_on_texts(df['clean_text'].values)

# Save tokenizer for the dashboard inference
os.makedirs('../models', exist_ok=True)
with open('../models/tokenizer.pkl', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

print("Tokenizer trained and saved.")

NameError: name 'Tokenizer' is not defined

## 3. Sequence Padding & Train/Test Split

In [ ]:
# Convert text to numeric sequences
X_seq = tokenizer.texts_to_sequences(df['clean_text'].values)

# pad sequences to fixed length
max_len = 40  # Max length of a cleaned tweet
X_pad = pad_sequences(X_seq, maxlen=max_len, padding='post', truncating='post')

# one-hot encode targets
y_cat = to_categorical(df['label'].values, num_classes=3)

# 80/20 train test split
X_train, X_test, y_train, y_test = train_test_split(X_pad, y_cat, test_size=0.2, random_state=42, stratify=df['label'].values)
print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

## 4. Building the LSTM Architecture

Our architecture consists of:
1. An **Embedding** layer for dense word representation.
2. A **SpatialDropout1D** layer.
3. A **Bidirectional LSTM** layer to extract context.
4. A hidden **Dense** layer.
5. A final **Dense** Softmax layer.

In [ ]:
embedding_dim = 128

model = Sequential([
    Embedding(input_dim=max_features, output_dim=embedding_dim, input_length=max_len),
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2)),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(3, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

## 5. Model Training

We use an **Early Stopping** callback to stop training if validation loss stops improving.

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

epochs = 8
batch_size = 64

print("Starting training...")
history = model.fit(
    X_train, y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

## 6. Model Evaluation & Loss Curves

In [ ]:
# Plot accuracy and loss curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(history.history['loss'], label='Train Loss')
ax1.plot(history.history['val_loss'], label='Val Loss')
ax1.set_title('Loss Curve')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend()

# Accuracy
ax2.plot(history.history['accuracy'], label='Train Accuracy')
ax2.plot(history.history['val_accuracy'], label='Val Accuracy')
ax2.set_title('Évolution de l\'Exactitude (Accuracy)')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')
ax2.legend()

plt.savefig('../data/learning_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# Predict on test set
y_pred_probs = model.predict(X_test)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

# Inverse mapping
target_names = ['negative', 'neutral', 'positive']

print("\nClassification Report: LSTM")
print(classification_report(y_true_classes, y_pred_classes, target_names=target_names))

In [ ]:
# Confusion Matrix
plt.figure(figsize=(7, 6))
cm = confusion_matrix(y_true_classes, y_pred_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(cmap='Purples', ax=plt.gca())
plt.title('Confusion Matrix: LSTM')
plt.savefig('../data/confusion_matrix_lstm.png', bbox_inches='tight')
plt.show()

## 7. Saving the LSTM Model

In [ ]:
# save modern keras model format
model.save('../models/lstm_model.keras')
print("Saved Keras LSTM model to '../models/lstm_model.keras'.")